# Definition-driven experiments and reproducible identity

This standalone lesson requires Python 3.10-3.13 and the matching installed DRYML version with its `sklearn` extra (`dryml[sklearn]`); that extra is not installed on Python 3.14. It runs offline with tiny fixed NumPy arrays and uses only temporary Store state. It replaces historical generated-ID loops with immutable nested definitions whose explicit estimator `random_state` is a reproducible replicate coordinate.

A `Definition` is an immutable structural expression. A `ConcreteDefinition` (CDef) is the exact materializable identity used by Repo and Store operations. Runtime training mutates fitted model and experiment state, but it does not rewrite either definition.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
from sklearn.ensemble import RandomForestRegressor

from dryml.core2 import Definition, Repo, SKIP_ARGS
from dryml.core2.store import DirStore
from dryml.data import ArrayDataset
from dryml.metrics import mean_squared_error
from dryml.models import Experiment
from dryml.models.sklearn import BasicTraining, RegressionModel

## Compose one nested template

Every identity-relevant value is constructor data on a maintained public class. The sklearn estimator exposes `random_state`; DRYML does not need an invented trial or replicate field. `n_jobs=1` and a small fixed forest keep execution bounded and deterministic.

In [ ]:
x_train = np.array(
    [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0],
     [1.0, 1.0], [2.0, 0.0], [2.0, 1.0]],
    dtype=np.float32,
)
y_train = np.array([0.0, 1.0, 1.0, 2.0, 2.0, 3.0], dtype=np.float32)

dataset_definition = Definition(ArrayDataset, (x_train, y_train))
model_template = Definition(
    RegressionModel,
    RandomForestRegressor,
    n_estimators=5,
    max_depth=3,
    random_state=11,
    n_jobs=1,
)
training_definition = Definition(BasicTraining)
experiment_template = Definition(
    Experiment,
    model_template,
    training_definition,
    train_data=dataset_definition,
)

## Copy-on-write variants

`with_kwarg` returns a new model definition, and `with_arg` installs it into a new experiment definition. Neither operation changes its source. Repeating the same template produces structural equality; changing the explicit seed changes the nested expression.

In [ ]:
same_template = Definition(
    Experiment,
    model_template,
    training_definition,
    train_data=dataset_definition,
)
seed_29_model = model_template.with_kwarg('random_state', 29)
seed_29_experiment = experiment_template.with_arg(0, seed_29_model)

assert model_template.kwargs['random_state'] == 11
assert experiment_template.args[0] == model_template
assert seed_29_model.kwargs['random_state'] == 29
assert experiment_template == same_template
assert experiment_template != seed_29_experiment

## Concretize and order exact identities

Identical templates concretize to one CDef identity. The two explicit seed values produce two distinct CDefs. Their stable hashes are used only as reproducible ordering keys; this lesson does not hard-code a particular hash value.

In [ ]:
template_cdef = experiment_template.concretize()
same_template_cdef = same_template.concretize()
assert template_cdef == same_template_cdef
assert template_cdef.stable_hash() == same_template_cdef.stable_hash()

variant_definitions = (
    (11, experiment_template),
    (29, seed_29_experiment),
)
ordered_variants = tuple(sorted(
    ((seed, definition.concretize()) for seed, definition in variant_definitions),
    key=lambda item: item[1].stable_hash(),
))
variant_cdefs = tuple(cdef for _, cdef in ordered_variants)

assert len(variant_cdefs) == 2
assert len(set(variant_cdefs)) == 2
assert [cdef.stable_hash() for cdef in variant_cdefs] == sorted(
    cdef.stable_hash() for cdef in variant_cdefs
)

## Reuse policy is not identity

A default Repo build reuses a live cached object for the same CDef. `instance='new', cache='none'` instead asks for fresh Python instances while retaining that same definition. The explicit pairing is required; asking for a new instance with a cache is rejected. Fresh instances are a materialization choice, not new persisted trial identities.

In [ ]:
materialization_repo = Repo()
try:
    reused_first = materialization_repo.load_or_build(template_cdef)
    reused_second = materialization_repo.load_or_build(template_cdef)
    assert reused_first is reused_second

    try:
        materialization_repo.load_or_build(template_cdef, instance='new')
    except ValueError as error:
        assert "cache='none'" in str(error)
    else:
        raise AssertionError('new instances with a cache should fail')

    fresh_first = materialization_repo.load_or_build(
        template_cdef, instance='new', cache='none'
    )
    fresh_second = materialization_repo.load_or_build(
        template_cdef, instance='new', cache='none'
    )
    assert fresh_first is not fresh_second
    assert fresh_first.model is not fresh_second.model
    assert fresh_first.definition == fresh_second.definition == template_cdef
finally:
    materialization_repo.close(flush=False)

## Train, save, and query the bounded set

Persistence uses normal Repo materialization so the complete live graph is available to save. Exactly two variants train. Each metric must be finite, and summaries remain in stable CDef order. The explicit `.stored().defs()` terminal asks only for saved definitions matching the shared structure: an `Experiment` with an `ArrayDataset`. The two roots remain distinct rather than colliding. The Store is a same-version temporary demonstration, not a cross-version interchange artifact.

In [ ]:
shared_support = Definition(
    Experiment,
    SKIP_ARGS,
    train_data=Definition(ArrayDataset, SKIP_ARGS),
)
summary = []

with TemporaryDirectory() as temporary_root:
    store_path = Path(temporary_root) / 'variant-store'
    training_repo = Repo(stores=DirStore(store_path, query_index='memory'))
    try:
        for seed, cdef in ordered_variants:
            experiment = training_repo.load_or_build(cdef)
            experiment.train()
            metric = mean_squared_error(
                experiment.model,
                experiment.train_data,
                batch_size=len(x_train),
            )
            if not np.isfinite(metric):
                raise ValueError('variant metric must be finite')
            training_repo.save_object(experiment)
            summary.append({
                'identity': cdef.stable_hash(),
                'random_state': seed,
                'mse': round(float(metric), 12),
            })

        queried_cdefs = tuple(
            training_repo.query(shared_support).stored().defs()
        )
        assert queried_cdefs == variant_cdefs
        assert len(set(queried_cdefs)) == 2
    finally:
        training_repo.close(flush=True)

assert len(summary) == 2
assert [item['identity'] for item in summary] == sorted(
    item['identity'] for item in summary
)
print('DEFINITION_VARIANT_SUMMARY=' + json.dumps(summary, sort_keys=True))

The durable identity lesson is deliberately narrower than the historical model-generation tutorial: vary supported constructor data, compare CDefs, and persist those definitions. Notebook-written helper modules, compute decorators, generated IDs, and multi-framework benchmark claims are intentionally omitted.